# Practical 01 — Python for AI - Environment, Data Structures and the Toolkit

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 1 |
| **Related lectures** | Lectures 1 and 2 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Set up and verify the Python environment used throughout this course.
2. Use the data structures that every later practical depends on: lists, dictionaries, sets and tuples.
3. Write and use generators, and explain how they differ from lists.
4. Represent an agent's percepts, actions and belief state in Python.
5. Implement the sense-update-decide-act loop from Lecture 2.

## 2. Background theory

Lecture 2 described an agent as running a loop: it senses the environment, updates a bounded summary of the past called the **belief state**, decides on an action from that summary alone, and acts. Today you build that loop in Python.

The programming ideas here are not incidental. Every later practical uses dictionaries to map situations to values, sets to record what has already been visited, and generators to produce candidate moves one at a time rather than building an enormous list in memory. If these are unfamiliar now, the search practicals in weeks 4 and 5 will be much harder than they need to be.

**Key points to keep in mind while you work:**

- A belief state is whatever the agent remembers; anything not in it cannot influence its behaviour.
- A generator produces values on demand, so it never builds the whole sequence in memory.
- A dictionary lookup is fast regardless of size, which is why we use them for state tables.
- A set membership test is also fast, which is why we use sets to record visited states.

## 3. Apparatus and setup

**Software:** Python 3.10 or later, Jupyter Notebook or JupyterLab, NumPy, Matplotlib.

Run the cell below first. If any import fails, install the missing package with `pip install <package>` and run the cell again before continuing.

In [1]:
import sys, subprocess, importlib

# Auto-install any missing package into THIS kernel's Python, then import it.
for pkg in ('numpy', 'matplotlib'):
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        print('Installing missing package:', pkg)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

print('Python', sys.version.split()[0])

import numpy, matplotlib
print('numpy', numpy.__version__)
print('matplotlib', matplotlib.__version__)
print('\nEnvironment ready.')

Python 3.14.5
numpy 2.5.1
matplotlib 3.11.2

Environment ready.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — The data structures you will actually use

Run this and read the output carefully. Each of these four structures is used for a specific purpose in later practicals, noted in the comments.

In [2]:
# A list: an ordered sequence you can add to. Used for the frontier in search.
frontier = ['A', 'B', 'C']
frontier.append('D')
print('frontier      :', frontier)

# A dictionary: a lookup table. Used for mapping states to values.
distance = {'A': 0, 'B': 3, 'C': 7}
print('distance to B :', distance['B'])

# A set: fast membership testing. Used to record what we have already seen.
visited = {'A', 'B'}
print('seen C?       :', 'C' in visited)

# A tuple: a fixed group of values. Used for coordinates and immutable states.
position = (3, 5)
print('position      :', position, 'row', position[0])

frontier      : ['A', 'B', 'C', 'D']
distance to B : 3
seen C?       : False
position      : (3, 5) row 3


### Step 2 — Generators: producing values one at a time

A generator computes each value only when it is asked for. In search problems the list of possible next moves can be enormous, so producing them one at a time matters. Note what happens when you consume a generator twice.

In [3]:
def squares_list(n):
    return [x * x for x in range(n)]        # builds the whole list at once

def squares_gen(n):
    for x in range(n):
        yield x * x                          # produces one value at a time

print('list   :', squares_list(5))
print('gen    :', list(squares_gen(5)))

g = squares_gen(5)
print('first sum :', sum(g))
print('second sum:', sum(g), '   <- a generator is consumed once')

list   : [0, 1, 4, 9, 16]
gen    : [0, 1, 4, 9, 16]
first sum : 30
second sum: 0    <- a generator is consumed once


### Step 3 — An agent and its environment

Here is a minimal agent in a corridor of rooms. It can move left or right, it senses only whether the current room is dirty, and its belief state records which rooms it has already cleaned. Read the loop and identify the four stages from Lecture 2.

In [4]:
import random
random.seed(0)

class Environment:
    def __init__(self, n_rooms=5):
        self.n = n_rooms
        self.dirty = {r: random.choice([True, False]) for r in range(n_rooms)}
        self.position = 0

    def percept(self):
        # The agent sees ONLY whether the current room is dirty.
        return {'dirty': self.dirty[self.position]}

    def act(self, action):
        if action == 'clean':
            self.dirty[self.position] = False
        elif action == 'right' and self.position < self.n - 1:
            self.position += 1
        elif action == 'left' and self.position > 0:
            self.position -= 1

env = Environment()
print('true state (hidden from the agent):', env.dirty)
print('what the agent can see           :', env.percept())

true state (hidden from the agent): {0: False, 1: False, 2: True, 3: False, 4: False}
what the agent can see           : {'dirty': False}


### Step 4 — The sense-update-decide-act loop

This agent keeps a belief state: the set of rooms it believes it has cleaned. Notice that the decision uses only the belief state and the current percept.

In [5]:
def run_agent(env, steps=12, verbose=True):
    cleaned = set()                       # THE BELIEF STATE
    for t in range(steps):
        percept = env.percept()           # SENSE
        if percept['dirty']:              # UPDATE + DECIDE
            action = 'clean'
            cleaned.add(env.position)
        else:
            action = 'right'
        if verbose:
            print('t=%2d  room=%d  percept=%-5s  action=%-5s  belief=%s'
                  % (t, env.position, percept['dirty'], action, sorted(cleaned)))
        env.act(action)                   # ACT
    return cleaned

env = Environment()
cleaned = run_agent(env)
print('\nrooms still dirty:', [r for r, d in env.dirty.items() if d])

t= 0  room=0  percept=False  action=right  belief=[]
t= 1  room=1  percept=False  action=right  belief=[]
t= 2  room=2  percept=False  action=right  belief=[]
t= 3  room=3  percept=False  action=right  belief=[]
t= 4  room=4  percept=True   action=clean  belief=[4]
t= 5  room=4  percept=False  action=right  belief=[4]
t= 6  room=4  percept=False  action=right  belief=[4]
t= 7  room=4  percept=False  action=right  belief=[4]
t= 8  room=4  percept=False  action=right  belief=[4]
t= 9  room=4  percept=False  action=right  belief=[4]
t=10  room=4  percept=False  action=right  belief=[4]
t=11  room=4  percept=False  action=right  belief=[4]

rooms still dirty: []


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Predict before you run

Write a generator that produces the first n even numbers. Before running it, write in a comment what you expect `sum()` to return the second time it is called on the same generator, and then check.

> **Marks:** 2

In [6]:
# My prediction for the second sum: 0.
# A generator is consumed the first time sum() walks through it. The second
# sum() call gets an already-exhausted generator, so there is nothing left to add.

def even_numbers(n):
    """Generate the first n even numbers: 2, 4, ..., 2n (one at a time)."""
    for i in range(1, n + 1):
        yield 2 * i

print('first 5 even numbers:', list(even_numbers(5)))

g = even_numbers(5)
print('first sum :', sum(g))
print('second sum:', sum(g), '   <- 0, exactly as predicted: the generator was already consumed')

first 5 even numbers: [2, 4, 6, 8, 10]
first sum : 30
second sum: 0    <- 0, exactly as predicted: the generator was already consumed


### Task 2 — Give the agent a better policy

The agent above always moves right, so it walks off the end of the corridor and stops making progress. Write a new policy that uses the belief state to turn around when it reaches the end. Run it and confirm every room is cleaned.

> **Marks:** 4

In [7]:
def run_agent_v2(env, steps=60, verbose=True):
    """Corridor agent whose policy turns around at the ends.
    It decides using ONLY its belief state + the current percept."""
    n = env.n                                   # prior knowledge: corridor length
    belief = {'pos': env.position,              # where I think I am (dead reckoning)
              'direction': 'right',             # which way I am heading
              'checked': set()}                 # rooms I have already inspected/cleaned
    cleaned = set()
    for t in range(steps):
        percept = env.percept()                 # SENSE
        pos = belief['pos']
        belief['checked'].add(pos)              # UPDATE belief
        if percept['dirty']:                    # DECIDE
            action = 'clean'
            cleaned.add(pos)
        elif len(belief['checked']) == n:
            action = 'stop'                     # every room inspected -> all clean
        else:
            if belief['direction'] == 'right' and pos == n - 1:
                belief['direction'] = 'left'    # hit the right end -> turn around
            elif belief['direction'] == 'left' and pos == 0:
                belief['direction'] = 'right'   # hit the left end -> turn around
            action = belief['direction']
        if verbose:
            print('t=%2d  room=%d  percept=%-5s  action=%-5s  checked=%s'
                  % (t, pos, percept['dirty'], action, sorted(belief['checked'])))
        if action == 'stop':
            break
        env.act(action)                         # ACT
        if action == 'right': belief['pos'] += 1
        elif action == 'left': belief['pos'] -= 1
    return cleaned

# Run 1: start in room 0 (same setup as the procedure)
random.seed(0)
env = Environment()
print('start dirty rooms:', [r for r, d in env.dirty.items() if d])
run_agent_v2(env)
print('rooms still dirty:', [r for r, d in env.dirty.items() if d])
print('Every room cleaned?', not any(env.dirty.values()))

# Run 2: start in the MIDDLE so the agent must hit an end and turn around
random.seed(0)
env = Environment()
env.position = 2
print('\n--- start in room 2: the agent has to reverse at the right end ---')
run_agent_v2(env)
print('Every room cleaned?', not any(env.dirty.values()))

start dirty rooms: [2]
t= 0  room=0  percept=False  action=right  checked=[0]
t= 1  room=1  percept=False  action=right  checked=[0, 1]
t= 2  room=2  percept=True   action=clean  checked=[0, 1, 2]
t= 3  room=2  percept=False  action=right  checked=[0, 1, 2]
t= 4  room=3  percept=False  action=right  checked=[0, 1, 2, 3]
t= 5  room=4  percept=False  action=stop   checked=[0, 1, 2, 3, 4]
rooms still dirty: []
Every room cleaned? True

--- start in room 2: the agent has to reverse at the right end ---
t= 0  room=2  percept=True   action=clean  checked=[2]
t= 1  room=2  percept=False  action=right  checked=[2]
t= 2  room=3  percept=False  action=right  checked=[2, 3]
t= 3  room=4  percept=False  action=left   checked=[2, 3, 4]
t= 4  room=3  percept=False  action=left   checked=[2, 3, 4]
t= 5  room=2  percept=False  action=left   checked=[2, 3, 4]
t= 6  room=1  percept=False  action=left   checked=[1, 2, 3, 4]
t= 7  room=0  percept=False  action=stop   checked=[0, 1, 2, 3, 4]
Every room cle

### Task 3 — Break the belief state deliberately

Remove the belief state from your agent — that is, stop recording which rooms have been cleaned — and run it again. Describe in a comment what changes about its behaviour, and explain why in terms of Lecture 2.

> **Marks:** 4

In [8]:
def run_agent_no_belief(env, steps=60, verbose=True, rng=None):
    """Same corridor, but NO belief state: no 'cleaned' set, no direction, no position.
    The action depends only on the current percept (a simple reflex agent)."""
    rng = rng or random.Random(1)
    first_all_clean = None                      # recorded by me (the observer), NOT by the agent
    for t in range(steps):
        percept = env.percept()                 # SENSE
        if percept['dirty']:                    # DECIDE from the percept alone
            action = 'clean'
        else:
            action = rng.choice(['left', 'right'])   # nothing to remember -> no sense of direction
        if verbose and t < 20:
            print('t=%2d  room=%d  percept=%-5s  action=%s' % (t, env.position, percept['dirty'], action))
        env.act(action)                         # ACT
        if first_all_clean is None and not any(env.dirty.values()):
            first_all_clean = t + 1
    return first_all_clean

random.seed(0)
env = Environment()
needed = run_agent_no_belief(env, steps=60)
print('...')
print('All rooms clean after', needed, 'actions - but the agent does not know that, so it keeps acting until step 60.')

# WHAT CHANGES (explanation in terms of Lecture 2):
# 1. It never stops. Cleaned and dirty-but-already-handled rooms give the same percept
#    ({'dirty': False}), so with no belief state it cannot tell "finished" from "not finished yet".
#    The agent with a belief state stopped after its own count of actions; this one wastes actions forever.
# 2. It has no direction or position, so it cannot sweep the corridor or turn around at the ends;
#    it can only wander randomly, re-visiting rooms it already cleaned. Walking becomes a random walk,
#    so it takes far more actions to reach every room.
# 3. Lecture 2: the agent decides from its belief state + current percept. The belief state is the ONLY
#    summary of the past. Remove it and the past cannot influence behaviour at all - the same percept
#    always produces the same kind of action, whatever has already happened.

t= 0  room=0  percept=False  action=left
t= 1  room=0  percept=False  action=left
t= 2  room=0  percept=False  action=right
t= 3  room=1  percept=False  action=left
t= 4  room=0  percept=False  action=right
t= 5  room=1  percept=False  action=right
t= 6  room=2  percept=True   action=clean
t= 7  room=2  percept=False  action=right
t= 8  room=3  percept=False  action=right
t= 9  room=4  percept=False  action=left
t=10  room=3  percept=False  action=left
t=11  room=2  percept=False  action=right
t=12  room=3  percept=False  action=left
t=13  room=2  percept=False  action=right
t=14  room=3  percept=False  action=right
t=15  room=4  percept=False  action=left
t=16  room=3  percept=False  action=right
t=17  room=4  percept=False  action=right
t=18  room=4  percept=False  action=left
t=19  room=3  percept=False  action=left
...
All rooms clean after 7 actions - but the agent does not know that, so it keeps acting until step 60.


### Task 4 — Count the work

Modify your agent to count how many actions it takes to clean every room. Run it on corridors of 5, 10 and 20 rooms and record the counts in the observations table.

> **Marks:** 3

In [9]:
def run_agent_v3(env, max_steps=1000):
    """run_agent_v2 + an action counter. Returns the number of actions (clean/left/right)
    taken before the agent knows every room is clean."""
    n = env.n
    pos, direction, checked = env.position, 'right', set()
    actions = 0                                  # NEW: count the work
    for t in range(max_steps):
        percept = env.percept()
        checked.add(pos)
        if percept['dirty']:
            action = 'clean'
        elif len(checked) == n:
            break                                # done - 'stop' is not counted as work
        else:
            if direction == 'right' and pos == n - 1: direction = 'left'
            elif direction == 'left' and pos == 0:    direction = 'right'
            action = direction
        env.act(action)
        actions += 1
        if action == 'right': pos += 1
        elif action == 'left': pos -= 1
    return actions

def actions_no_belief(env, max_steps=5000, seed=1):
    """Actions the belief-less agent needs until every room is actually clean (observer's count)."""
    rng = random.Random(seed)
    for t in range(max_steps):
        if not any(env.dirty.values()):
            return t
        if env.percept()['dirty']:
            env.act('clean')
        else:
            env.act(rng.choice(['left', 'right']))
    return None

def make_env(n, seed=0):
    random.seed(seed)
    return Environment(n)

print('%-6s %-22s %-22s %s' % ('rooms', 'with belief state', 'without belief state', 'every room cleaned?'))
results = {}
for n in (5, 10, 20):
    e1 = make_env(n); with_b = run_agent_v3(e1)
    e2 = make_env(n); without_b = actions_no_belief(e2)
    ok = (not any(e1.dirty.values())) and (not any(e2.dirty.values()))
    results[n] = (with_b, without_b, ok)
    print('%-6d %-22d %-22d %s' % (n, with_b, without_b, 'Yes' if ok else 'No'))

# Extra: average over 300 random corridors to see the general trend (single runs are noisy)
import statistics
print('\naverage over 300 random corridors:')
for n in (5, 10, 20):
    A = [run_agent_v3(make_env(n, s)) for s in range(300)]
    B = [actions_no_belief(make_env(n, s), seed=s) for s in range(300)]
    print('n=%2d   with belief: %6.1f   without belief: %6.1f' % (n, statistics.mean(A), statistics.mean(B)))

rooms  with belief state      without belief state   every room cleaned?
5      5                      7                      Yes
10     11                     72                     Yes
20     27                     522                    Yes



average over 300 random corridors:
n= 5   with belief:    6.5   without belief:   17.9
n=10   with belief:   14.0   without belief:   85.8
n=20   with belief:   29.1   without belief:  367.3


### Task 5 — Describe your agent in four parts

In a markdown cell below, describe your agent using the four parts from Lecture 1: abilities, goals, prior knowledge, and experience. Be specific — name the actual variables in your code where you can.

> **Marks:** 2

In [10]:
# My four-part description is in the markdown cell below.
# Quick look at the variables it refers to (the agent's belief state):
print("belief state = {'pos': ..., 'direction': ..., 'checked': set(...)}  (inside run_agent_v2 / v3)")

belief state = {'pos': ..., 'direction': ..., 'checked': set(...)}  (inside run_agent_v2 / v3)


**Four-part description of my agent (Lecture 1)**

- **Abilities (what it can do):** it can perceive whether the current room is dirty (`env.percept()`), and act with `'clean'`, `'left'`, `'right'` (or `'stop'` in `run_agent_v2`) through `env.act(action)`.
- **Goals (what it wants):** every room in the corridor clean, using as few actions as possible (`actions` counter in `run_agent_v3`), and stopping once it knows the job is finished.
- **Prior knowledge (what it knows in advance):** the corridor length `env.n` (stored as `n`), its starting room `env.position`, and the rules of its world (rooms in a line, walls at both ends, `'clean'` always works).
- **Experience (what it learns while running):** the percept `percept['dirty']` at each step and its belief state: `belief['pos']` (where it is), `belief['direction']` (which way it is heading) and `belief['checked']` / `checked` (rooms already inspected/cleaned). It uses this to turn around at the ends and to know when every room is done.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Corridor size | Actions taken (with belief state) | Actions taken (without) | Every room cleaned? |
|---|---|---|---|
| 5 rooms  | 5 | 7 | Yes |
| 10 rooms | 11 | 72 | Yes |
| 20 rooms | 27 | 522 | Yes |

*(Single seeded run per corridor size, `random.seed(0)` for the dirt layout. "Without" counts the actions until every room was actually clean, as seen by me the observer, since the agent without a belief state never knows when to stop. Averages over 300 random corridors are printed in the Task 4 cell: with belief about 6.5 / 14.0 / 29.1, without belief about 17.9 / 85.8 / 367.3.)*

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. What exactly does your agent's belief state contain, and what did you deliberately leave out of it?
2. Your agent senses only the current room. Give a situation where two different rooms produce identical percepts but require different actions.
3. How did the number of actions grow as the corridor got longer? Was the growth roughly proportional, or worse?

*Your answers:*

1. The belief state is a small dictionary: `pos` (the room the agent thinks it is in), `direction` (the way it is heading) and `checked` (the set of rooms it has already inspected and cleaned). I deliberately left out the full history of percepts and actions, the time step, the original dirt layout and the true state `env.dirty`. Anything not in the belief state cannot influence behaviour, and the agent does not need those to finish the job.

2. Room 0 and room 4 (the two ends) when both are clean give the identical percept `{'dirty': False}`. At room 0 the right action is to move right, at room 4 it is to turn around (move left) or stop. Likewise, a clean room that is still unchecked and a clean room that was already visited look the same, but one needs more searching and the other does not. The percept alone cannot separate them, only the belief state can.

3. With the belief state the number of actions grew roughly in proportion to the corridor length (5 → 5, 10 → 11, 20 → 27 in the seeded runs; averages 6.5 → 14.0 → 29.1): about n - 1 moves plus one clean per dirty room. Without it the growth was much worse than proportional: doubling the corridor multiplied the average actions by about 4 to 5 (17.9 → 85.8 → 367.3), roughly quadratic, because the agent just random-walks instead of sweeping.

## 8. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P01_<your-roll-number>.ipynb`

In [11]:
NAME = "Ali Hassan Babar"
ROLL_NUMBER = "2025-CSE-005"
print(NAME, ROLL_NUMBER)

Ali Hassan Babar 2025-CSE-005
